In [24]:
import requests
import pandas as pd
import time
from urllib.parse import unquote


url = "http://apis.data.go.kr/1613000/RouteCongestionLevel/getRouteCongestionLevel"
service_key = unquote('9d8d0134bf818cc4e30c699c4f659e33d28b19b4cd6ceae9f990213260b4a50c')

def collect_data():
    all_items = []
    target_dates = ['20240101', '20240105', '20240107']
    pages_to_collect = 10  # 날짜당 10페이지씩 수집 (총 1000건씩)

    for date in target_dates:
        print(f"\n{date} 수집 시작...")

        for page in range(1, pages_to_collect + 1):
            params = {
                'serviceKey': service_key,
                'pageNo': str(page),      # 페이지 번호 변경
                'numOfRows': '100',       # 최대치인 100으로 고정
                'dataType': 'JSON',
                'opr_ymd': date,
                'ctpv_cd': '26',
                'sgg_cd': '26350'
            }

            try:
                response = requests.get(url, params=params)
                data = response.json()

                if 'Response' in data and 'body' in data['Response']:
                    items = data['Response']['body']['items']['item']
                    if items:
                        items_list = items if isinstance(items, list) else [items]
                        all_items.extend(items_list)
                        # 현재 수집 현황 출력
                        if page % 5 == 0: # 5페이지마다 로그 출력
                            print(f"\t-> {page}페이지 완료 (누적 {len(all_items)}건)")
                    else:
                        break # 더 이상 데이터가 없으면 다음 날짜로

                time.sleep(0.2)

            except Exception as e:
                print(f"   -> ❌ {date} {page}페이지 에러: {e}")
                break

    # 2. 결과 저장
    if all_items:
        df = pd.DataFrame(all_items)
        df.to_csv("busan_bus_raw.csv", index=False)
        print(f"[MP1] 총 {len(df)}건 수집")
    else:
        print("수집된 데이터 없음")

collect_data()


20240101 수집 시작...
	-> 5페이지 완료 (누적 500건)
	-> 10페이지 완료 (누적 1000건)

20240105 수집 시작...
	-> 5페이지 완료 (누적 1500건)
	-> 10페이지 완료 (누적 2000건)

20240107 수집 시작...
	-> 5페이지 완료 (누적 2500건)
	-> 10페이지 완료 (누적 3000건)
[MP1] 총 3000건 수집
